# 04 — Model 1: SigLIP2 + Cross-Attention Fusion (Improvement Phase)

After Stages A/B/C (all naive concat+MLP fusion) failed to beat `text_only_bert` (0.704 ROC-AUC / 0.212 PR-AUC vs. best SigLIP2 result of 0.659/0.184 in Stage C), this combines three changes at once, run via `configs/exp04_model1_crossattn.yaml`:

1. **Cross-attention fusion** instead of concat+MLP -- text pools over image patch tokens and image pools over text tokens (two-way), before the final head. Concat fusion never let text and image actually condition on each other.
2. **Partial unfreeze (top 2 layers) + LoRA together** -- Stage B (full unfreeze, 57M params) overfit almost immediately; Stage C (LoRA alone, ~986K params) was stable but plateaued around PR-AUC 0.184. This targets a middle ground (~32.4M trainable params, 8.5%).
3. **Full-resolution-only training data** -- `src/data/filter_fullres.py` drops the ~39% of rows whose thumbnail is a small CDN fallback tier (82x62 or 232x175px), keeping 53,544/87,334 train rows. Isolates whether inconsistent image quality was part of the problem.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt


## 1. Run Model 1 training

Subprocess (`!`) so all real output is captured in this cell and saved with the notebook -- also writes `experiments/siglip2_model1_crossattn_fullres/train_log.csv` and `val_log.csv` incrementally (survives even if this cell/kernel is interrupted).

**Do not run this at the same time as any other GPU/MPS process** (another training run, `image_only`/`title_image_frozen`/`text_only_bert` baselines, etc.) -- running two MPS-heavy processes concurrently caused a real hang earlier in this project that needed a full machine restart to clear. Run things one at a time.

In [ ]:
!cd .. && python3 -u -m src.training.train --config configs/exp04_model1_crossattn.yaml


## 2. Load the logged curves and plot

In [ ]:
run_dir = "../experiments/siglip2_model1_crossattn_fullres"

train_log = pd.read_csv(f"{run_dir}/train_log.csv")
val_log = pd.read_csv(f"{run_dir}/val_log.csv")

print("Training steps logged:", len(train_log))
print("Validation epochs logged:", len(val_log))
val_log


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_log["step"], train_log["loss"])
axes[0].set_xlabel("step")
axes[0].set_ylabel("train loss")
axes[0].set_title("Model 1 training loss")

axes[1].plot(val_log["epoch"], val_log["pr_auc"], marker="o", label="PR-AUC")
axes[1].plot(val_log["epoch"], val_log["roc_auc"], marker="o", label="ROC-AUC")
axes[1].axhline(0.212, color="gray", linestyle="--", label="text_only_bert PR-AUC (0.212)")
axes[1].axhline(0.184, color="lightgray", linestyle=":", label="Stage C PR-AUC (0.184)")
axes[1].set_xlabel("epoch")
axes[1].set_title("Model 1 validation metrics")
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Compare against every other model tried

Final numbers as recorded in the experiment log (`project_explanation_HE.md` section 11) -- Model 1's row is read live from `val_log.csv` above rather than hardcoded.

In [ ]:
best_model1 = val_log.loc[val_log["pr_auc"].idxmax()]

results = pd.DataFrame([
    {"model": "tfidf_logreg",        "roc_auc": 0.660, "pr_auc": 0.208},
    {"model": "text_only_bert",       "roc_auc": 0.704, "pr_auc": 0.212},
    {"model": "image_only",           "roc_auc": 0.619, "pr_auc": 0.148},
    {"model": "title_image_frozen",   "roc_auc": 0.619, "pr_auc": 0.147},
    {"model": "siglip2_stage_a",      "roc_auc": 0.651, "pr_auc": 0.167},
    {"model": "siglip2_stage_c_lora", "roc_auc": 0.659, "pr_auc": 0.184},
    {"model": "model1_crossattn",     "roc_auc": best_model1["roc_auc"], "pr_auc": best_model1["pr_auc"]},
]).set_index("model")

results.sort_values("pr_auc", ascending=False)
